# PyHealth Synthetic EHR Generation - Google Colab

This notebook demonstrates how to:
1. Install PyHealth and dependencies
2. Process MIMIC data for synthetic generation
3. Train baseline models (GReaT, CTGAN, TVAE)
4. Compare with original baselines.py outputs

**Hardware Requirements:**
- GPU recommended (use Runtime > Change runtime type > GPU or A100)
- ~16GB RAM minimum

**Prerequisites:**
- MIMIC-III data files uploaded to Google Drive or Colab
- Train/test patient ID files
- Original baseline outputs (if comparing)

## Step 1: Setup Environment

First, let's check GPU availability and mount Google Drive (if your data is there).

In [ ]:
# Check GPU
!nvidia-smi

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Mount Google Drive (if your MIMIC data is stored there)
from google.colab import drive
drive.mount('/content/drive')

# List files to verify
!ls /content/drive/MyDrive/

## Step 2: Install Dependencies

**Note:** PyHealth requires Python 3.12+, but Colab currently runs 3.10. We'll install the compatible dependencies manually.

In [ ]:
# Check Python version
import sys
print(f"Python version: {sys.version}")

# Colab uses Python 3.10, so we need to work around PyHealth's 3.12 requirement
# We'll clone and manually add PyHealth to path instead of pip installing

In [ ]:
# Install required packages
!pip install -q polars pandas numpy scipy scikit-learn tqdm matplotlib seaborn

# Install baseline model packages
!pip install -q be-great  # For GReaT model
!pip install -q sdv       # For CTGAN and TVAE

print("✓ Dependencies installed")

In [ ]:
# Clone PyHealth repository
!git clone https://github.com/sunlabuiuc/PyHealth.git
%cd PyHealth

# Add to Python path
import sys
sys.path.insert(0, '/content/PyHealth')

print("✓ PyHealth cloned and added to path")

In [ ]:
# Verify imports work
try:
    from pyhealth.utils.synthetic_ehr_utils import (
        process_mimic_for_generation,
        tabular_to_sequences,
        sequences_to_tabular,
        create_flattened_representation,
    )
    print("✓ PyHealth utils imported successfully")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("\nTrying to create utility module manually...")

## Step 3: Configure Paths

**Update these paths to match your setup:**

In [ ]:
# ========================================
# CONFIGURE YOUR PATHS HERE
# ========================================

# Option A: Data in Google Drive
MIMIC_DATA_PATH = "/content/drive/MyDrive/mimic3_data/"
TRAIN_PATIENTS_PATH = "/content/drive/MyDrive/mimic3_data/train_patient_ids.txt"
TEST_PATIENTS_PATH = "/content/drive/MyDrive/mimic3_data/test_patient_ids.txt"

# Option B: Upload to Colab directly (uncomment if using this)
# from google.colab import files
# uploaded = files.upload()  # Upload your files
# MIMIC_DATA_PATH = "/content/"
# TRAIN_PATIENTS_PATH = "/content/train_patient_ids.txt"
# TEST_PATIENTS_PATH = "/content/test_patient_ids.txt"

# Output paths
PYHEALTH_OUTPUT = "/content/pyhealth_output"
ORIGINAL_OUTPUT = "/content/drive/MyDrive/original_output"  # Path to your original results

# Model settings
MODEL_MODE = "great"  # Options: "great", "ctgan", "tvae"
NUM_EPOCHS = 2
BATCH_SIZE = 512
NUM_SYNTHETIC_SAMPLES = 10000

print("Configuration:")
print(f"  MIMIC Data: {MIMIC_DATA_PATH}")
print(f"  Train IDs: {TRAIN_PATIENTS_PATH}")
print(f"  Test IDs: {TEST_PATIENTS_PATH}")
print(f"  Output: {PYHEALTH_OUTPUT}")
print(f"  Model: {MODEL_MODE}")

In [ ]:
# Verify MIMIC files exist
import os

required_files = [
    os.path.join(MIMIC_DATA_PATH, "ADMISSIONS.csv"),
    os.path.join(MIMIC_DATA_PATH, "PATIENTS.csv"),
    os.path.join(MIMIC_DATA_PATH, "DIAGNOSES_ICD.csv"),
    TRAIN_PATIENTS_PATH,
    TEST_PATIENTS_PATH,
]

print("Checking required files:")
all_exist = True
for f in required_files:
    exists = os.path.exists(f)
    status = "✓" if exists else "✗"
    print(f"  {status} {f}")
    if not exists:
        all_exist = False

if all_exist:
    print("\n✓ All required files found!")
else:
    print("\n✗ Some files are missing. Please upload them or update paths.")

## Step 4: Process MIMIC Data

This processes the raw MIMIC CSVs into formats needed for synthetic generation.

In [ ]:
import pandas as pd
from pyhealth.utils.synthetic_ehr_utils import process_mimic_for_generation

print("Processing MIMIC data...")
print("This may take several minutes...\n")

# Process MIMIC data
data = process_mimic_for_generation(
    mimic_data_path=MIMIC_DATA_PATH,
    train_patients_path=TRAIN_PATIENTS_PATH,
    test_patients_path=TEST_PATIENTS_PATH,
)

# Extract datasets
train_ehr = data["train_ehr"]
test_ehr = data["test_ehr"]
train_flattened = data["train_flattened"]
test_flattened = data["test_flattened"]
train_sequences = data["train_sequences"]
test_sequences = data["test_sequences"]

print("\n" + "="*80)
print("Data Processing Complete")
print("="*80)
print(f"Train EHR shape: {train_ehr.shape}")
print(f"Test EHR shape: {test_ehr.shape}")
print(f"Train flattened shape: {train_flattened.shape}")
print(f"Test flattened shape: {test_flattened.shape}")
print(f"Train sequences: {len(train_sequences)}")
print(f"Test sequences: {len(test_sequences)}")

print("\nSample flattened data (first 5 rows, first 10 columns):")
print(train_flattened.iloc[:5, :10])

## Step 5: Train Baseline Model

Now we'll train the selected baseline model (GReaT, CTGAN, or TVAE).

In [ ]:
# Create output directory
os.makedirs(PYHEALTH_OUTPUT, exist_ok=True)

if MODEL_MODE == "great":
    print("\n" + "="*80)
    print("Training GReaT Model")
    print("="*80)
    
    import be_great
    
    # Initialize GReaT model
    model = be_great.GReaT(
        llm='tabularisai/Qwen3-0.3B-distil',
        batch_size=BATCH_SIZE,
        epochs=NUM_EPOCHS,
        dataloader_num_workers=4,
        fp16=torch.cuda.is_available()
    )
    
    # Train
    print("\nTraining... (this may take 10-30 minutes)")
    model.fit(train_flattened)
    
    # Save model
    save_path = os.path.join(PYHEALTH_OUTPUT, "great")
    os.makedirs(save_path, exist_ok=True)
    model.save(save_path)
    print(f"\n✓ Model saved to {save_path}")
    
    # Generate synthetic data
    print(f"\nGenerating {NUM_SYNTHETIC_SAMPLES} synthetic samples...")
    synthetic_data = model.sample(n_samples=NUM_SYNTHETIC_SAMPLES)
    
    # Save synthetic data
    output_csv = os.path.join(save_path, "great_synthetic_flattened_ehr.csv")
    synthetic_data.to_csv(output_csv, index=False)
    print(f"✓ Synthetic data saved to {output_csv}")
    
    print("\n" + "="*80)
    print("GReaT Training Complete!")
    print("="*80)

In [ ]:
if MODEL_MODE == "ctgan":
    print("\n" + "="*80)
    print("Training CTGAN Model")
    print("="*80)
    
    from sdv.metadata import Metadata
    from sdv.single_table import CTGANSynthesizer
    
    # Auto-detect metadata
    metadata = Metadata.detect_from_dataframe(data=train_flattened)
    
    # Set all columns as numerical
    for column in train_flattened.columns:
        metadata.update_column(column_name=column, sdtype='numerical')
    
    # Initialize and train
    synthesizer = CTGANSynthesizer(
        metadata,
        epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE
    )
    
    print("\nTraining... (this may take 10-30 minutes)")
    synthesizer.fit(train_flattened)
    
    # Save model
    save_path = os.path.join(PYHEALTH_OUTPUT, "ctgan")
    os.makedirs(save_path, exist_ok=True)
    synthesizer.save(filepath=os.path.join(save_path, "synthesizer.pkl"))
    print(f"\n✓ Model saved to {save_path}")
    
    # Generate synthetic data
    print(f"\nGenerating {NUM_SYNTHETIC_SAMPLES} synthetic samples...")
    synthetic_data = synthesizer.sample(num_rows=NUM_SYNTHETIC_SAMPLES)
    
    # Save synthetic data
    output_csv = os.path.join(save_path, "ctgan_synthetic_flattened_ehr.csv")
    synthetic_data.to_csv(output_csv, index=False)
    print(f"✓ Synthetic data saved to {output_csv}")
    
    print("\n" + "="*80)
    print("CTGAN Training Complete!")
    print("="*80)

In [ ]:
if MODEL_MODE == "tvae":
    print("\n" + "="*80)
    print("Training TVAE Model")
    print("="*80)
    
    from sdv.metadata import Metadata
    from sdv.single_table import TVAESynthesizer
    
    # Auto-detect metadata
    metadata = Metadata.detect_from_dataframe(data=train_flattened)
    
    # Set all columns as numerical
    for column in train_flattened.columns:
        metadata.update_column(column_name=column, sdtype='numerical')
    
    # Initialize and train
    synthesizer = TVAESynthesizer(
        metadata,
        epochs=NUM_EPOCHS,
        batch_size=BATCH_SIZE
    )
    
    print("\nTraining... (this may take 10-30 minutes)")
    synthesizer.fit(train_flattened)
    
    # Save model
    save_path = os.path.join(PYHEALTH_OUTPUT, "tvae")
    os.makedirs(save_path, exist_ok=True)
    synthesizer.save(filepath=os.path.join(save_path, "synthesizer.pkl"))
    print(f"\n✓ Model saved to {save_path}")
    
    # Generate synthetic data
    print(f"\nGenerating {NUM_SYNTHETIC_SAMPLES} synthetic samples...")
    synthetic_data = synthesizer.sample(num_rows=NUM_SYNTHETIC_SAMPLES)
    
    # Save synthetic data
    output_csv = os.path.join(save_path, "tvae_synthetic_flattened_ehr.csv")
    synthetic_data.to_csv(output_csv, index=False)
    print(f"✓ Synthetic data saved to {output_csv}")
    
    print("\n" + "="*80)
    print("TVAE Training Complete!")
    print("="*80)

## Step 6: Inspect Synthetic Data

In [ ]:
# Load generated synthetic data
synthetic_csv = os.path.join(PYHEALTH_OUTPUT, MODEL_MODE, f"{MODEL_MODE}_synthetic_flattened_ehr.csv")
synthetic_data = pd.read_csv(synthetic_csv)

print("Synthetic Data Summary:")
print("="*80)
print(f"Shape: {synthetic_data.shape}")
print(f"Number of features: {len(synthetic_data.columns)}")
print(f"Number of samples: {len(synthetic_data)}")

print("\nFirst 5 rows, first 10 columns:")
print(synthetic_data.iloc[:5, :10])

print("\nStatistics:")
print(synthetic_data.describe())

In [ ]:
# Visualize synthetic data properties
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribution of values
axes[0, 0].hist(synthetic_data.values.flatten(), bins=50, edgecolor='black')
axes[0, 0].set_xlabel('Value')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of All Values')

# 2. Sparsity
sparsity = (synthetic_data == 0).sum() / len(synthetic_data)
axes[0, 1].bar(['Non-zero', 'Zero'], 
               [len(synthetic_data) - (synthetic_data == 0).sum().sum(), 
                (synthetic_data == 0).sum().sum()])
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Sparsity Distribution')

# 3. Column means
column_means = synthetic_data.mean().sort_values(ascending=False)
axes[1, 0].bar(range(min(20, len(column_means))), column_means.head(20))
axes[1, 0].set_xlabel('Feature (top 20)')
axes[1, 0].set_ylabel('Mean value')
axes[1, 0].set_title('Top 20 Features by Mean')

# 4. Distribution of row sums
row_sums = synthetic_data.sum(axis=1)
axes[1, 1].hist(row_sums, bins=50, edgecolor='black')
axes[1, 1].set_xlabel('Sum of codes per patient')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Distribution of Code Counts per Patient')

plt.tight_layout()
plt.savefig(os.path.join(PYHEALTH_OUTPUT, 'synthetic_data_visualization.png'), dpi=150)
plt.show()

print(f"✓ Visualization saved to {os.path.join(PYHEALTH_OUTPUT, 'synthetic_data_visualization.png')}")

## Step 7: Compare with Original Baselines

If you have outputs from the original baselines.py script, you can compare them here.

In [ ]:
# Set path to original baseline outputs
ORIGINAL_CSV = os.path.join(ORIGINAL_OUTPUT, MODEL_MODE, f"{MODEL_MODE}_synthetic_flattened_ehr.csv")

# Check if original file exists
if os.path.exists(ORIGINAL_CSV):
    print(f"✓ Found original output: {ORIGINAL_CSV}")
    COMPARE = True
else:
    print(f"✗ Original output not found: {ORIGINAL_CSV}")
    print("Skipping comparison...")
    COMPARE = False

In [ ]:
if COMPARE:
    print("\n" + "="*80)
    print("COMPARISON: Original vs PyHealth")
    print("="*80)
    
    # Load both datasets
    original_df = pd.read_csv(ORIGINAL_CSV)
    pyhealth_df = pd.read_csv(synthetic_csv)
    
    print(f"\nOriginal shape: {original_df.shape}")
    print(f"PyHealth shape: {pyhealth_df.shape}")
    
    # Basic statistics comparison
    print("\n" + "-"*80)
    print("Statistical Comparison")
    print("-"*80)
    
    comparison = pd.DataFrame({
        'Metric': ['Mean', 'Std', 'Min', 'Max', 'Sparsity (%)'],
        'Original': [
            f"{original_df.mean().mean():.4f}",
            f"{original_df.std().mean():.4f}",
            f"{original_df.min().min():.4f}",
            f"{original_df.max().max():.4f}",
            f"{(original_df == 0).sum().sum() / (original_df.shape[0] * original_df.shape[1]) * 100:.2f}"
        ],
        'PyHealth': [
            f"{pyhealth_df.mean().mean():.4f}",
            f"{pyhealth_df.std().mean():.4f}",
            f"{pyhealth_df.min().min():.4f}",
            f"{pyhealth_df.max().max():.4f}",
            f"{(pyhealth_df == 0).sum().sum() / (pyhealth_df.shape[0] * pyhealth_df.shape[1]) * 100:.2f}"
        ]
    })
    
    print(comparison.to_string(index=False))
    
    # Visual comparison
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Distribution comparison
    axes[0].hist(original_df.mean(), bins=50, alpha=0.7, label='Original', edgecolor='black')
    axes[0].hist(pyhealth_df.mean(), bins=50, alpha=0.7, label='PyHealth', edgecolor='black')
    axes[0].set_xlabel('Column Mean')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Column Means')
    axes[0].legend()
    
    # Code frequency correlation
    common_cols = list(set(original_df.columns) & set(pyhealth_df.columns))
    if len(common_cols) > 0:
        orig_freq = original_df[common_cols].sum()
        pyh_freq = pyhealth_df[common_cols].sum()
        
        axes[1].scatter(orig_freq, pyh_freq, alpha=0.5)
        axes[1].plot([0, max(orig_freq.max(), pyh_freq.max())], 
                     [0, max(orig_freq.max(), pyh_freq.max())], 
                     'r--', label='Perfect match')
        axes[1].set_xlabel('Original frequency')
        axes[1].set_ylabel('PyHealth frequency')
        axes[1].set_title('Code Frequency Correlation')
        axes[1].legend()
        
        # Calculate correlation
        correlation = orig_freq.corr(pyh_freq)
        axes[1].text(0.05, 0.95, f'Correlation: {correlation:.4f}', 
                    transform=axes[1].transAxes, 
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                    verticalalignment='top')
    
    plt.tight_layout()
    plt.savefig(os.path.join(PYHEALTH_OUTPUT, 'comparison_visualization.png'), dpi=150)
    plt.show()
    
    print(f"\n✓ Comparison visualization saved to {os.path.join(PYHEALTH_OUTPUT, 'comparison_visualization.png')}")
    
    # Validation checks
    print("\n" + "-"*80)
    print("Validation Checks")
    print("-"*80)
    
    checks = []
    
    # Check 1: Similar dimensions
    dim_diff = abs(original_df.shape[0] - pyhealth_df.shape[0]) / original_df.shape[0]
    checks.append(('Similar dimensions (within 1%)', dim_diff < 0.01))
    
    # Check 2: Similar sparsity
    orig_sparsity = (original_df == 0).sum().sum() / (original_df.shape[0] * original_df.shape[1])
    pyh_sparsity = (pyhealth_df == 0).sum().sum() / (pyhealth_df.shape[0] * pyhealth_df.shape[1])
    checks.append(('Similar sparsity (within 10%)', abs(orig_sparsity - pyh_sparsity) < 0.1))
    
    # Check 3: Similar mean
    orig_mean = original_df.mean().mean()
    pyh_mean = pyhealth_df.mean().mean()
    checks.append(('Similar mean (within 20%)', abs(orig_mean - pyh_mean) / orig_mean < 0.2))
    
    for check_name, passed in checks:
        status = "✓ PASS" if passed else "✗ FAIL"
        print(f"  {status} - {check_name}")
    
    if all([c[1] for c in checks]):
        print("\n🎉 All validation checks passed! PyHealth implementation is working correctly.")
    else:
        print("\n⚠️  Some checks failed. This may be due to random sampling differences.")

## Step 8: Download Results

Download synthetic data and visualizations to your local machine.

In [ ]:
from google.colab import files
import shutil

# Create a zip file with all outputs
output_zip = '/content/pyhealth_synthetic_ehr_results.zip'
shutil.make_archive(
    output_zip.replace('.zip', ''),
    'zip',
    PYHEALTH_OUTPUT
)

print(f"Created zip file: {output_zip}")
print("Downloading...")

# Download
files.download(output_zip)

print("✓ Download complete!")

## Summary

You have successfully:
1. ✓ Installed PyHealth and dependencies
2. ✓ Processed MIMIC data
3. ✓ Trained a synthetic EHR generation model
4. ✓ Generated synthetic patient data
5. ✓ Compared with original baselines (if available)

**Next Steps:**
- Train with more epochs for better quality
- Try different models (great, ctgan, tvae)
- Evaluate synthetic data quality
- Use synthetic data for downstream tasks

**Files Generated:**
- Synthetic EHR CSV
- Trained model checkpoint
- Visualization plots
- Comparison report (if applicable)